In [1]:
# Kill all processess on GPU
!fuser -v /dev/nvidia* -k

                     USER        PID ACCESS COMMAND
/dev/nvidia0:        root      26869 F...m python3
/dev/nvidiactl:      root      26869 F...m python3
/dev/nvidia-uvm:     root      26869 F...m python3


In [2]:
# Check GPU status
!nvidia-smi

Sat Jul 11 12:03:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   76C    P0             35W /   70W |       0MiB /  15360MiB |     36%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# Libraries

In [3]:
%%capture
!uv pip uninstall torchao torchaudio torchvision -y
!uv pip install \
  "transformers==4.53.3" \
  "peft==0.17.1" \
  "trl" \
  "accelerate" \
  "bitsandbytes" \
  "wandb"

In [4]:
import torch
from datetime import datetime
from transformers import AutoModelForQuestionAnswering, AutoTokenizer
from peft import PeftModel

# Configurations

In [5]:
# Run configuration
SRC_LANG = 'en'
TGT_LANG = 'vi'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Model configuration
# MODEL_ID = 'alxxtexxr/XLM-R-Base-squad-en-LoRA-Merged-v260623145250'
# LORA_ID = 'alxxtexxr/XLM-R-Base-wikipedia-vi-LoRA-v260622154525'
MODEL_ID = 'alxxtexxr/XLM-R-Base-squad-en-5K-LoRA-Merged-v260711104723'
LORA_ID = 'alxxtexxr/XLM-R-Base-wikipedia-vi-5K-LoRA-v260711111235'

assert SRC_LANG in MODEL_ID and TGT_LANG in LORA_ID, "Model and LoRA IDs do not match the specified source language and target language."

hub_merged_model_id = f"{MODEL_ID.split(SRC_LANG)[0]}{TGT_LANG}-LoRA-Addition-v{datetime.now().strftime("%y%m%d%H%M%S")}"
print(f"Hub merged model ID: {hub_merged_model_id}")

Hub merged model ID: alxxtexxr/XLM-R-Base-squad-vi-LoRA-Addition-v260711120329


# Model

In [6]:
# Load base model
base_model = AutoModelForQuestionAnswering.from_pretrained(MODEL_ID)

# Load LoRA model
lora_model = PeftModel.from_pretrained(base_model, LORA_ID)

# Merge LoRA model into base model
merged_model = lora_model.merge_and_unload()
merged_model.to(DEVICE).eval()

print("device:", merged_model.device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


adapter_config.json:   0%|          | 0.00/892 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/10.7M [00:00<?, ?B/s]

device: cuda:0


In [ ]:
# Upload the merged model to Hugging Face
merged_model.push_to_hub(hub_merged_model_id)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.push_to_hub(hub_merged_model_id)

print(f"Merged model uploaded to: https://huggingface.co/{hub_merged_model_id}")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...gxa7u3a/model.safetensors:   4%|4         | 47.1MB / 1.11GB            

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...1/sentencepiece.bpe.model: 100%|##########| 5.07MB / 5.07MB            

  ...mpls9ac6w1/tokenizer.json: 100%|##########| 17.1MB / 17.1MB            

Merged model uploaded to: https://huggingface.co/alxxtexxr/XLM-R-Base-squad-vi-LoRA-Addition-v260711120329


: 